## Example 1: blosc2 functionality test 1

In [ ]:
import blosc2
import numpy as np

# Example 1: Basic compression and decompression with blosc2
print("=== Blosc2 Basic Compression Example ===")

try:
    # Create sample data
    data = b"Hello Blosc2! " * 1000
    print(f"Original data size: {len(data)} bytes")
    
    # Compress data using blosc2
    compressed = blosc2.compress(data)
    print(f"Compressed size: {len(compressed)} bytes")
    print(f"Compression ratio: {len(data)/len(compressed):.2f}:1")
    
    # Decompress data
    decompressed = blosc2.decompress(compressed)
    print(f"Decompressed size: {len(decompressed)} bytes")
    print(f"Data integrity verified: {data == decompressed}")
    
    # Show version info
    print(f"Blosc2 version: {blosc2.__version__}")
    print("Basic compression/decompression completed successfully!")
    
except Exception as e:
    print(f"Error in basic compression: {e}")
    raise AssertionError(
        f"blosc2 example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: blosc2 functionality test 2

In [ ]:
import blosc2
import numpy as np
import time

# Example 2: NDArray creation and operations with blosc2
print("=== Blosc2 NDArray Operations Example ===")

try:
    # Create a large numpy array
    np_array = np.random.rand(1000, 1000).astype(np.float32)
    print(f"NumPy array shape: {np_array.shape}")
    print(f"NumPy array size: {np_array.nbytes / 1024 / 1024:.2f} MB")
    
    # Create a blosc2 NDArray with compression
    start_time = time.time()
    b2_array = blosc2.asarray(np_array, chunks=(100, 100), blocks=(50, 50))
    creation_time = time.time() - start_time
    
    print(f"Blosc2 NDArray created in {creation_time:.4f} seconds")
    print(f"Chunks shape: {b2_array.chunks}")
    print(f"Blocks shape: {b2_array.blocks}")
    print(f"Compression ratio: {b2_array.cratio:.2f}:1")
    
    # Perform operations
    start_time = time.time()
    result = b2_array.sum()
    op_time = time.time() - start_time
    
    print(f"Sum operation completed in {op_time:.4f} seconds")
    print(f"Array sum: {result:.2f}")
    print("NDArray operations completed successfully!")
    
except Exception as e:
    print(f"Error in NDArray operations: {e}")
    raise AssertionError(
        f"blosc2 example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: blosc2 functionality test 3

In [ ]:
import blosc2
import numpy as np

# Example 3: Advanced compression codecs and filters
print("=== Blosc2 Advanced Codecs and Filters Example ===")

try:
    # Create test data with patterns
    data = np.arange(10000, dtype=np.int32)
    print(f"Original data size: {data.nbytes} bytes")
    
    # Test different codecs (using proper Codec enum)
    codecs = [blosc2.Codec.LZ4, blosc2.Codec.LZ4HC, blosc2.Codec.ZLIB, blosc2.Codec.ZSTD, blosc2.Codec.BLOSCLZ]
    results = {}
    
    for codec in codecs:
        try:
            # Compress with specific codec
            compressed = blosc2.compress(data, codec=codec, clevel=5)
            ratio = data.nbytes / len(compressed)
            results[codec.name] = {
                'size': len(compressed),
                'ratio': ratio
            }
            print(f"{codec.name}: {len(compressed)} bytes, ratio {ratio:.2f}:1")
        except Exception as codec_error:
            print(f"{codec.name}: Not available - {codec_error}")
            raise
    
    # Test with filters for better compression (using proper Filter enum)
    print("\n--- Testing with different filters ---")
    
    filters_to_test = [
        (blosc2.Filter.NOFILTER, "No filter"),
        (blosc2.Filter.SHUFFLE, "Shuffle filter"),
        (blosc2.Filter.BITSHUFFLE, "Bit-shuffle filter"),
        (blosc2.Filter.DELTA, "Delta filter")
    ]
    
    for filter_enum, filter_name in filters_to_test:
        try:
            compressed_filtered = blosc2.compress(data, codec=blosc2.Codec.LZ4, filter=filter_enum)
            ratio_filtered = data.nbytes / len(compressed_filtered)
            print(f"{filter_name}: {len(compressed_filtered)} bytes, ratio {ratio_filtered:.2f}:1")
        except Exception as filter_error:
            print(f"{filter_name}: Error - {filter_error}")
            raise
    
    # Verify decompression
    test_compressed = blosc2.compress(data, codec=blosc2.Codec.LZ4, filter=blosc2.Filter.SHUFFLE)
    decompressed = blosc2.decompress(test_compressed)
    reconstructed = np.frombuffer(decompressed, dtype=np.int32)
    print(f"\nData integrity verified: {np.array_equal(data, reconstructed)}")
    
    # Show compression parameters information
    print(f"\n--- Compression Parameters Info ---")
    cparams = blosc2.CParams(codec=blosc2.Codec.LZ4, filters=[blosc2.Filter.SHUFFLE], clevel=5)
    print(f"CParams codec: {cparams.codec}")
    print(f"CParams filters: {cparams.filters}")
    print(f"CParams compression level: {cparams.clevel}")
    
    print("Advanced codecs and filters testing completed!")
    
except Exception as e:
    print(f"Error in advanced compression: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 4: blosc2 functionality test 4

In [ ]:
import blosc2
import numpy as np
import tempfile
import os

# Example 4: Persistent storage with Blosc2 containers
print("=== Blosc2 Persistent Storage Example ===")

try:
    # Create sample data
    data1 = np.random.rand(500, 500).astype(np.float64)
    data2 = np.arange(1000000).astype(np.int64)
    
    # Create a temporary file for storage
    with tempfile.NamedTemporaryFile(suffix='.b2nd', delete=False) as temp_file:
        temp_path = temp_file.name
    
    print(f"Creating persistent container at: {temp_path}")
    
    # Create a blosc2 container for persistent storage
    array_container = blosc2.zeros(
        shape=(1000, 1000), 
        dtype=np.float64,
        chunks=(100, 100),
        blocks=(50, 50),
        urlpath=temp_path,
        mode='w'
    )
    
    # Store data in container
    array_container[:500, :500] = data1
    print(f"Stored {data1.size} float64 values")
    print(f"Container shape: {array_container.shape}")
    print(f"Container size on disk: {os.path.getsize(temp_path)} bytes")
    print(f"Compression ratio: {array_container.cratio:.2f}:1")
    
    # Read data back from container
    read_data = array_container[:500, :500]
    print(f"Data integrity verified: {np.allclose(data1, read_data)}")
    
    # Demonstrate additional container operations
    print(f"\n--- Container Information ---")
    print(f"Container dtype: {array_container.dtype}")
    print(f"Container chunks: {array_container.chunks}")
    print(f"Container blocks: {array_container.blocks}")
    print(f"Compressed bytes: {array_container.cbytes}")
    
    # Clean up properly (NDArray objects don't have close method, just delete them)
    del array_container  # This properly releases the array
    
    # Verify file exists and clean up
    if os.path.exists(temp_path):
        print(f"Container file size: {os.path.getsize(temp_path)} bytes")
        os.unlink(temp_path)
        print("Temporary file cleaned up")
    
    print("Persistent storage example completed successfully!")
    
except Exception as e:
    print(f"Error in persistent storage: {e}")
    import traceback
    traceback.print_exc()
    
    # Cleanup in case of error
    if 'temp_path' in locals() and os.path.exists(temp_path):
        try:
            os.unlink(temp_path)
            print("Cleaned up temporary file after error")
        except:
            pass
            raise
    raise

## Example 5: blosc2 functionality test 5

In [ ]:
import blosc2
import numpy as np
import time

# Example 5: Comprehensive Blosc2 capabilities and performance overview
print("=== Blosc2 Comprehensive Capabilities Overview ===")

try:
    print("Blosc2 Library Information:")
    print(f"Version: {blosc2.__version__}")
    
    # Show compressor list properly
    compressors = blosc2.compressor_list()
    print(f"Available compressors: {[codec.name for codec in compressors]}")
    
    # Performance comparison with different data types
    print("\n--- Performance Comparison ---")
    data_types = [
        ('Random floats', np.random.rand(100000).astype(np.float32)),
        ('Sequential integers', np.arange(100000, dtype=np.int32)),
        ('Repeated pattern', np.tile(np.arange(100), 1000).astype(np.int32))
    ]
    
    for name, data in data_types:
        print(f"\n{name}:")
        print(f"  Original size: {data.nbytes} bytes")
        
        try:
            # Time compression (using proper codec enum)
            start_time = time.time()
            compressed = blosc2.compress(data, codec=blosc2.Codec.LZ4, clevel=5)
            comp_time = time.time() - start_time
            
            # Time decompression
            start_time = time.time()
            decompressed = blosc2.decompress(compressed)
            decomp_time = time.time() - start_time
            
            ratio = data.nbytes / len(compressed)
            comp_speed = data.nbytes / (1024 * 1024) / comp_time if comp_time > 0 else float('inf')
            decomp_speed = data.nbytes / (1024 * 1024) / decomp_time if decomp_time > 0 else float('inf')
            
            print(f"  Compressed size: {len(compressed)} bytes")
            print(f"  Compression ratio: {ratio:.2f}:1")
            print(f"  Compression speed: {comp_speed:.1f} MB/s")
            print(f"  Decompression speed: {decomp_speed:.1f} MB/s")
            
        except Exception as perf_error:
            print(f"  Performance test error: {perf_error}")
            raise
    
    # Codec comparison
    print(f"\n--- Codec Comparison ---")
    test_data = np.arange(50000, dtype=np.int32)
    
    codecs_to_test = [blosc2.Codec.LZ4, blosc2.Codec.LZ4HC, blosc2.Codec.ZSTD, blosc2.Codec.BLOSCLZ]
    
    for codec in codecs_to_test:
        try:
            start_time = time.time()
            compressed = blosc2.compress(test_data, codec=codec, clevel=5)
            comp_time = time.time() - start_time
            ratio = test_data.nbytes / len(compressed)
            
            print(f"  {codec.name}: {ratio:.2f}:1 ratio, {comp_time*1000:.1f}ms")
            
        except Exception as codec_error:
            print(f"  {codec.name}: Error - {codec_error}")
            raise
    
    # Filter effectiveness demonstration
    print(f"\n--- Filter Effectiveness ---")
    structured_data = np.array([(i, i*2, i*3) for i in range(10000)], 
                              dtype=[('a', 'i4'), ('b', 'i4'), ('c', 'i4')])
    
    filters_to_compare = [
        (blosc2.Filter.NOFILTER, "No filter"),
        (blosc2.Filter.SHUFFLE, "Shuffle"),
        (blosc2.Filter.BITSHUFFLE, "Bit-shuffle"),
        (blosc2.Filter.DELTA, "Delta")
    ]
    
    for filter_enum, filter_name in filters_to_compare:
        try:
            compressed = blosc2.compress(structured_data, codec=blosc2.Codec.LZ4, filter=filter_enum)
            ratio = structured_data.nbytes / len(compressed)
            print(f"  {filter_name}: {ratio:.2f}:1 ratio")
        except Exception as filter_error:
            print(f"  {filter_name}: Error - {filter_error}")
            raise
    
    print("\n--- Key Blosc2 Features ---")
    print("✓ High-performance compression with multiple codecs")
    print("✓ N-dimensional array support with chunking")
    print("✓ Persistent storage capabilities")
    print("✓ Advanced filters (shuffle, delta, bitshuffle) for better compression")
    print("✓ Multi-threading support for parallel processing")
    print("✓ Memory-efficient operations on large datasets")
    print("✓ Integration with NumPy and scientific computing ecosystem")
    print("✓ Backwards compatibility with Blosc v1")
    print("✓ Enhanced metadata and extensibility")
    
    print(f"\n--- Configuration Information ---")
    print(f"Max threads: {blosc2.nthreads}")
    print(f"Available filters: {[f.name for f in blosc2.Filter]}")
    print(f"Available codecs: {[c.name for c in blosc2.Codec]}")
    
    print("\nBlosc2 comprehensive overview completed successfully!")
    
except Exception as e:
    print(f"Error in comprehensive overview: {e}")
    import traceback
    traceback.print_exc()
    raise